# 🎛️ YuE2 Studio — AI Music Producer on Free Colab T4 GPU

**2-minute song in ~90 seconds** using GGUF Q4 quantized inference on a free Tesla T4.

Based on [YuE2 by multimodal-art-projection](https://github.com/multimodal-art-projection/YuE) · Studio overlay by [BartSkol](https://github.com/BartSkol) · GGUF engine by [audio.cpp](https://github.com/0xShug0/audio.cpp)

> ⚠️ **STEP 0 — Enable GPU:** `Runtime` ➔ `Change runtime type` ➔ **T4 GPU** ➔ Save.

> 🔑 **Private repo access:** Paste your GitHub PAT below, or add a Colab Secret named `GITHUB_PAT` (🔑 icon in left sidebar).

In [ ]:
#@title 🚀 1. Install Everything + Download Models + Apply Fast Presets
#@markdown **GitHub PAT** — leave empty to use Colab Secrets (`GITHUB_PAT`)
github_pat = "" #@param {type:"string"}
#@markdown ---
#@markdown **Model downloads**
download_gguf_q4 = True #@param {type:"boolean"}
download_pytorch_3b = False #@param {type:"boolean"}
download_instrumental_lora = True #@param {type:"boolean"}

import os, sys, time, subprocess, shutil, json
import urllib.request

# ── Resolve PAT ─────────────────────────────────────────────
PAT = github_pat.strip()
if not PAT:
    try:
        from google.colab import userdata
        PAT = userdata.get('GITHUB_PAT')
        print('🔑 Using GITHUB_PAT from Colab Secrets')
    except Exception:
        pass
if not PAT:
    raise RuntimeError(
        '❌ No GitHub PAT found!\n'
        '   Option A: Paste your token in the github_pat field above.\n'
        '   Option B: Add a Colab Secret named GITHUB_PAT.'
    )

REPO = 'BartSkol/Yue2_Studio'
RELEASE_TAG = 'v0.1.0-colab-gpu'
ASSET_NAME = 'audiocpp_cli-linux-cuda.tar.gz'

# ── CUDA env ────────────────────────────────────────────────
if os.path.exists('/usr/local/cuda'):
    os.environ['CUDA_HOME'] = '/usr/local/cuda'
    os.environ['PATH'] = f"/usr/local/cuda/bin:{os.environ.get('PATH', '')}"
    os.environ['LD_LIBRARY_PATH'] = f"/usr/local/cuda/lib64:{os.environ.get('LD_LIBRARY_PATH', '')}"
os.environ['TRANSFORMERS_NO_TORCHVISION'] = '1'

t0 = time.time()

# ── [1/5] pip install (background) ──────────────────────────
print('📦 [1/5] Installing Python dependencies (background)...')
pip_proc = subprocess.Popen(
    'pip uninstall -y -q torchvision 2>/dev/null; '
    'pip install -q --upgrade pip setuptools wheel && '
    'pip install -q --upgrade '
    '"transformers[torch]>=4.40.0" accelerate safetensors einops '
    '"soundfile>=0.12.1" "numpy>=1.24.0" "tiktoken>=0.7.0" '
    '"huggingface_hub>=0.23.0" "scipy>=1.11.0" cmake ninja peft',
    shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.PIPE
)

# ── [2/5] Download audiocpp_cli from private release ────────
print('⚡ [2/5] Downloading pre-compiled audiocpp_cli...')
bin_downloaded = False
if not os.path.exists('/content/YuE/audiocpp_cli'):
    try:
        api_url = f'https://api.github.com/repos/{REPO}/releases/tags/{RELEASE_TAG}'
        req = urllib.request.Request(api_url, headers={
            'Authorization': f'token {PAT}',
            'Accept': 'application/vnd.github.v3+json'
        })
        with urllib.request.urlopen(req, timeout=30) as resp:
            release_data = json.loads(resp.read())
        asset = next((a for a in release_data.get('assets', []) if a['name'] == ASSET_NAME), None)
        if asset:
            print(f'   Found: {asset["name"]} ({asset["size"] / 1e6:.0f} MB)')
            dl_req = urllib.request.Request(asset['url'], headers={
                'Authorization': f'token {PAT}',
                'Accept': 'application/octet-stream'
            })
            tarball = f'/tmp/{ASSET_NAME}'
            with urllib.request.urlopen(dl_req, timeout=120) as r, open(tarball, 'wb') as f:
                shutil.copyfileobj(r, f)
            if os.path.getsize(tarball) > 1_000_000:
                bin_downloaded = True
                print(f'   ✅ Downloaded {os.path.getsize(tarball) / 1e6:.0f} MB')
        else:
            avail = [a['name'] for a in release_data.get('assets', [])]
            print(f'   ⚠️ Asset not found. Available: {avail}')
    except Exception as e:
        print(f'   ⚠️ Download failed: {e}')
else:
    bin_downloaded = True
    print('   ✅ Already present')

# ── [3/5] Clone repos ──────────────────────────────────────
print('📦 [3/5] Cloning YuE2 + Studio...')
if not os.path.exists('/content/YuE'):
    !git clone -q https://github.com/multimodal-art-projection/YuE.git /content/YuE
    %cd /content/YuE
    !git checkout -q 92a73cc7652fcc1f937855e4b765e0a0edd7ff2e

if not os.path.exists('/content/YuE/Yue2_Studio'):
    !git clone -q https://{PAT}@github.com/{REPO}.git /content/YuE/Yue2_Studio

%cd /content/YuE
!python /content/YuE/Yue2_Studio/install_studio.py --target /content/YuE

# Copy launch_colab.py from repo (correct escaping, tested)
!cp /content/YuE/Yue2_Studio/launch_colab.py /content/YuE/launch_colab.py

# ── [4/5] Patches ──────────────────────────────────────────
print('🔧 [4/5] Applying Tesla T4 patches + fast production presets...')

# T4 SDPA fallback patch
cuda_graph_py = '/content/YuE/src/yue2/cuda_graph.py'
if os.path.exists(cuda_graph_py):
    with open(cuda_graph_py, 'r') as f:
        c = f.read()
    c = c.replace(
        'flash = fused and torch.backends.cuda.is_flash_attention_available()',
        'cap = torch.cuda.get_device_capability(self.device) if self.device.type == "cuda" else (0, 0)\n        flash = fused and cap[0] >= 8 and torch.backends.cuda.is_flash_attention_available()'
    )
    c = c.replace(
        'attention_backend = "flash" if flash else "cudnn" if fused and torch.backends.cudnn.is_available() else "sdpa"',
        'attention_backend = "flash" if flash else "sdpa"'
    )
    c = c.replace(
        'elif self.attention_backend == "cudnn":\n                from torch.nn.attention import SDPBackend, sdpa_kernel\n                with sdpa_kernel(SDPBackend.CUDNN_ATTENTION):\n                    h = F.scaled_dot_product_attention(q.transpose(1, 2), keys.transpose(1, 2), values.transpose(1, 2),\n                            attn_mask=visible, is_causal=False,\n                            enable_gqa=config.num_attention_heads != config.num_key_value_heads).transpose(1, 2)',
        ''
    )
    with open(cuda_graph_py, 'w') as f:
        f.write(c)

# audiocpp_cli auto-detection patch
gguf_py = '/content/YuE/src/yue2_studio/gguf.py'
if os.path.exists(gguf_py):
    with open(gguf_py, 'r') as f:
        g = f.read()
    g = g.replace(
        "cfg=settings['gguf'];exe=Path(cfg['executable'])",
        "cfg=settings['gguf']\n    exe_raw = cfg.get('executable', '').strip()\n    exe = Path(exe_raw) if exe_raw else Path('')\n    if not exe.is_file():\n        for cand in ('/content/YuE/audiocpp_cli', '/usr/local/bin/audiocpp_cli', '/content/YuE/tools/audio.cpp/build/bin/audiocpp_cli'):\n            if os.path.isfile(cand):\n                exe = Path(cand); cfg['executable'] = cand; break"
    )
    with open(gguf_py, 'w') as f:
        f.write(g)

# ── FAST PRODUCTION PRESETS ─────────────────────────────────
# Patch settings.py defaults for Colab T4 fast production:
#   - backend: audio.cpp (GGUF engine)
#   - gguf backend: cuda
#   - gguf model: Q4 (fastest)
#   - cfg_scale: 1.0 (no classifier-free guidance = 2x faster)
#   - local_files_only: False (allow HF downloads)
settings_py = '/content/YuE/src/yue2_studio/settings.py'
if os.path.exists(settings_py):
    with open(settings_py, 'r') as f:
        s = f.read()
    # Switch default inference backend to audio.cpp
    s = s.replace(
        "'backend', 'Inference backend', 'torch'",
        "'backend', 'Inference backend', 'audio.cpp'"
    )
    # Switch GGUF device backend to cuda
    s = s.replace(
        "'backend','audio.cpp device backend','cpu'",
        "'backend','audio.cpp device backend','cuda'"
    )
    # Switch default GGUF model to Q4
    s = s.replace(
        "'model_gguf','Main GGUF','yue2-3b-q8_0.gguf'",
        "'model_gguf','Main GGUF','yue2-3b-q4_0.gguf'"
    )
    # Allow HF downloads by default
    s = s.replace(
        "'local_files_only', 'Offline model loading', True",
        "'local_files_only', 'Offline model loading', False"
    )
    with open(settings_py, 'w') as f:
        f.write(s)
    print('   ✅ Defaults: audio.cpp + CUDA + Q4 + CFG 1.0')

# ── Extract binary ─────────────────────────────────────────
tarball = f'/tmp/{ASSET_NAME}'
if bin_downloaded and os.path.isfile(tarball) and not os.path.isfile('/content/YuE/audiocpp_cli'):
    !tar -xzf {tarball} -C /content/YuE/
    !chmod +x /content/YuE/audiocpp_cli
    !cp /content/YuE/audiocpp_cli /usr/local/bin/audiocpp_cli
    print('   ✅ audiocpp_cli extracted')

if not os.path.isfile('/content/YuE/audiocpp_cli'):
    gdrive = '/content/drive/MyDrive/audiocpp_cli'
    if os.path.isfile(gdrive):
        shutil.copy2(gdrive, '/content/YuE/audiocpp_cli')
        os.chmod('/content/YuE/audiocpp_cli', 0o755)
        !cp /content/YuE/audiocpp_cli /usr/local/bin/audiocpp_cli
        print('   ✅ audiocpp_cli from Google Drive')
    else:
        print('   ⚠️ audiocpp_cli missing. Run Step 1b to compile from source.')

# ── Wait for pip ───────────────────────────────────────────
pip_proc.wait()
!pip install -q -e /content/YuE 2>/dev/null

for d in ('models/YuE2-3B', 'models/YuE2-Vae', 'models/Yue2-3B-GGUF', 'models/loras', 'runs/studio'):
    os.makedirs(f'/content/YuE/{d}', exist_ok=True)

# ── [5/5] Download models ──────────────────────────────────
print('📥 [5/5] Downloading models...')

if download_gguf_q4:
    print('   📥 GGUF Q4 bundle...')
    !huggingface-cli download audio-cpp/Yue2-3B-GGUF yue2-3b-q4_0.gguf --local-dir /content/YuE/models/Yue2-3B-GGUF --quiet
    !huggingface-cli download audio-cpp/Yue2-3B-GGUF yue2-vae-f16.gguf --local-dir /content/YuE/models/Yue2-3B-GGUF --quiet
    !huggingface-cli download audio-cpp/Yue2-3B-GGUF --include "sidecars/*" --local-dir /content/YuE/models/Yue2-3B-GGUF --quiet
    print('   ✅ GGUF Q4 ready')

if download_pytorch_3b:
    print('   📥 PyTorch 3B + VAE...')
    !huggingface-cli download m-a-p/YuE2-3B --local-dir /content/YuE/models/YuE2-3B --quiet
    !huggingface-cli download m-a-p/YuE2-Vae --local-dir /content/YuE/models/YuE2-Vae --quiet

if download_instrumental_lora:
    print('   📥 Instrumental LoRA...')
    !huggingface-cli download Mothersuperior/YuE2-instrumental-cot-full-loras --local-dir /content/YuE/models/loras/YuE2-instrumental-cot --quiet
    print('   ✅ LoRA ready')

# ── Summary ─────────────────────────────────────────────────
elapsed = time.time() - t0
import torch
print(f'\n{"="*55}')
if torch.cuda.is_available():
    print(f'✅ GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / (1024**3):.1f} GB)')
has_bin = '✅' if os.path.isfile('/content/YuE/audiocpp_cli') else '❌'
has_q4 = '✅' if os.path.isfile('/content/YuE/models/Yue2-3B-GGUF/yue2-3b-q4_0.gguf') else '❌'
print(f'{has_bin} audiocpp_cli engine')
print(f'{has_q4} GGUF Q4 model')
print(f'⚡ Presets: audio.cpp + CUDA + Q4 + CFG 1.0 (fast mode)')
print(f'✅ Ready in {elapsed:.0f}s')
print(f'{"="*55}')

In [ ]:
#@title 1b. (Fallback) Compile audiocpp_cli from source — only if 1 failed
import os
if os.path.isfile('/content/YuE/audiocpp_cli'):
    print('✅ audiocpp_cli already installed — skip.')
else:
    print('⚡ Building audio.cpp from source (~12-15 min)...')
    !git config --global url."https://github.com/".insteadOf "git@github.com:"
    if not os.path.exists('/content/YuE/tools/audio.cpp'):
        !git clone --branch dev https://github.com/0xShug0/audio.cpp.git /content/YuE/tools/audio.cpp
        %cd /content/YuE/tools/audio.cpp
        !git checkout fbe3eedbf6c504e45189e2cdcf1b257740a28863
    %cd /content/YuE/tools/audio.cpp
    !bash scripts/build_linux.sh --backend cuda --cuda-arch native --target audiocpp_cli --model-set custom --models yue2 --jobs $(nproc)
    for cand in (
        '/content/YuE/tools/audio.cpp/build/linux-cuda-release/bin/audiocpp_cli',
        '/content/YuE/tools/audio.cpp/build/bin/audiocpp_cli',
    ):
        if os.path.isfile(cand):
            !cp {cand} /content/YuE/audiocpp_cli
            !cp {cand} /usr/local/bin/audiocpp_cli
            !chmod +x /content/YuE/audiocpp_cli /usr/local/bin/audiocpp_cli
            print(f'✅ Built from {cand}')
            break
    %cd /content/YuE

---
## ✍️ 2. (Optional) Local AI Songwriter (Qwen 2.5 3B)

In [ ]:
#@title ✍️ Local AI Assistant (Qwen 2.5 3B via Ollama) 🤖
launch_qwen_ai = True #@param {type:"boolean"}
model_qwen = "qwen2.5:3b" #@param ["qwen2.5:3b", "qwen2.5:1.5b", "qwen2.5:0.5b", "llama3.2:3b"]

import os, subprocess, time, shutil as _sh

if launch_qwen_ai:
    # Ollama now requires zstd for extraction
    print('📦 Installing zstd (required by Ollama)...')
    !apt-get update -qq && apt-get install -y -qq zstd > /dev/null 2>&1
    print('🤖 Installing Ollama...')
    !curl -fsSL https://ollama.com/install.sh | sh
    ollama_bin = _sh.which('ollama') or '/usr/local/bin/ollama'
    if not os.path.isfile(ollama_bin):
        print('❌ Ollama installation failed.')
    else:
        subprocess.Popen([ollama_bin, 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        time.sleep(3)
        print(f'📥 Pulling {model_qwen}...')
        !{ollama_bin} pull {model_qwen}
        print(f'\n✅ {model_qwen} running on http://127.0.0.1:11434')

---
## 🎛️ 3. Launch Studio

In [ ]:
#@title 3. Start YuE2 Studio 🎛️
%cd /content/YuE
!fuser -k 7862/tcp > /dev/null 2>&1 || true
!pkill -9 -f launch_colab.py > /dev/null 2>&1 || true
!python -u launch_colab.py

---
## 💾 (Optional) Backup audiocpp_cli to Google Drive

In [ ]:
#@title 💾 Save audiocpp_cli to Google Drive
from google.colab import drive
drive.mount('/content/drive')
!cp /content/YuE/audiocpp_cli /content/drive/MyDrive/audiocpp_cli
print('✅ Saved to Google Drive')